# ComplianceGPT Generator v1.0 (Multi-Version Unit Test)

**Role:** The experimental laboratory for the "Answerer" component.

**Objective:** Instantiate and verify the "Citation-Contracted Answerer" logic across NIST SP 800-53 Rev 4 and Rev 5 using real profile data. This version implements strict Python-side enforcement of ODP logic ("Traffic Cop") rather than relying solely on the LLM.

**Workflow:**
1.  **Setup:** Install Python dependencies (transformers, accelerate, bitsandbytes, pyyaml) and load **Qwen 2.5-7B-Instruct** in 4-bit quantized form via the Hugging Face transformers library.
2.  **Build Engine:** Dynamically write the `generator.py` logic module to the local filesystem. This module includes input sanitization, robust JSON parsing, and deterministic ODP substitution logic.
3.  **Test:** Execute unit tests using specific **Organization Profile YAML files** imported from Google Drive.

### Success Criteria
- **Test A (Rev 5):**
  - Input: Query about Account Review Frequency (AC-2).
  - Expected Output: `status: "OK"`, answer contains `"90 days"` (sourced from `org_profile_example_rev5.yaml`).
- **Test B (Rev 4):**
  - Input: Query about Access Control Policy Responsibility (AC-1).
  - Expected Output: `status: "OK"`, answer contains `"Security Officer"` (sourced from `org_profile_example_rev4.yaml`).
- **Test C (Missing Data):**
  - Input: Query with empty profile.
  - Expected Output: `status: "PARAMS_REQUIRED"`, `odp_required_list` contains missing key.


In [10]:
# 1. Environment Setup
try:
    import google.colab
    IN_COLAB = True
    print("Running in Colab: Installing dependencies...")
    get_ipython().system('pip install -q "transformers>=4.45.0" "accelerate>=0.33.0" bitsandbytes pyyaml')
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print("Running locally. Ensure dependencies are installed.")


Running in Colab: Installing dependencies...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# 2. Load Model (Qwen 2.5-7B-Instruct)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_ID}...")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Load Model (4-bit Quantization for optimized memory usage on T4 GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model Loaded.")


Loading Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model Loaded.


In [12]:
# 3. Load Generator Logic from shared script on Drive
# Instead of writing generator.py from a big string, we import the real file.

from pathlib import Path
import sys
import importlib.util

GENERATOR_PATH = Path("/content/drive/MyDrive/answerer_v0/generator/generator.py")

if not GENERATOR_PATH.exists():
    raise FileNotFoundError(
        f"generator.py not found at {GENERATOR_PATH}. "
        "Check that Google Drive is mounted and the path is correct."
    )

spec = importlib.util.spec_from_file_location("compliance_generator", str(GENERATOR_PATH))
module = module = importlib.util.module_from_spec(spec)
sys.modules["compliance_generator"] = module
spec.loader.exec_module(module)

ComplianceGenerator = module.ComplianceGenerator
load_org_profile = module.load_org_profile

print("Loaded ComplianceGenerator and load_org_profile from", GENERATOR_PATH)


Loaded ComplianceGenerator and load_org_profile from /content/drive/MyDrive/answerer_v0/generator/generator.py


In [13]:
# 4. Instantiate the Generator with the loaded model

generator = ComplianceGenerator(model, tokenizer)
print("Generator initialized.")


Generator initialized.


In [14]:
# 5. Execute Unit Tests
from pprint import pprint
import os

def verify_path(path):
    if not os.path.exists(path):
        print(f" WARNING: File not found at {path}. Check Google Drive.")
        return False
    return True

# ----------------------------------------------------------
# TEST A: NIST Rev 5 (Happy Path - Data Substitution)
# ----------------------------------------------------------
print("\n==========================================")
print("TEST A: NIST Rev 5 (Substitution Check)")
print("==========================================")

rev5_path = "/content/drive/MyDrive/answerer_v0/generator/ODP/rev5/org_profile_example_rev5.yaml"

if verify_path(rev5_path):
    rev5_profile = load_org_profile(rev5_path)
    # AC-2 query targeting 'ac-02_odp.05' (Expected: '90 days')
    query = "How often must account management reviews occur according to AC-2?"
    docs = [{
        "id": "NIST_SP800-53:rev5:AC-2",
        "text": "The organization reviews accounts for compliance {{ insert: param, ac-02_odp.05 }}."
    }]

    res = generator.generate(query, docs, rev5_profile)
    pprint(res)

    if "90 days" in res.get('answer_text', "") and res.get('status') == 'OK':
        print(" SUCCESS: Parameter substituted correctly.")
    else:
        print(" FAILURE: Substitution failed.")
else:
    print("CRITICAL ERROR: Rev 5 profile not found. TEST A was not executed.")

# ----------------------------------------------------------
# TEST B: NIST Rev 4 (Happy Path - Legacy Support)
# ----------------------------------------------------------
print("\n\n==========================================")
print("TEST B: NIST Rev 4 (Legacy Check)")
print("==========================================")

rev4_path = "/content/drive/MyDrive/answerer_v0/generator/ODP/rev4/org_profile_example_rev4.yaml"

if verify_path(rev4_path):
    rev4_profile = load_org_profile(rev4_path)
    # AC-1 query targeting 'ac-1_prm_1' (Expected: 'Security Officer')
    query = "Who is responsible for the Access Control Policy?"
    docs = [{
        "id": "NIST_SP800-53:rev4:AC-1",
        "text": "Develops, documents, and disseminates to {{ insert: param, ac-1_prm_1 }} the policy."
    }]

    res = generator.generate(query, docs, rev4_profile)
    pprint(res)

    if "Security Officer" in res.get('answer_text', "") and res.get('status') == 'OK':
        print(" SUCCESS: Parameter substituted correctly.")
    else:
        print(" FAILURE: Substitution failed.")
else:
    print("CRITICAL ERROR: Rev 4 profile not found. TEST B was not executed.")
# ----------------------------------------------------------
# TEST C: Negative Test (Traffic Cop Check)
# ----------------------------------------------------------
print("\n\n==========================================")
print("TEST C: Missing Data (Traffic Cop Check)")
print("==========================================")

# We explicitly pass an EMPTY profile to force a missing parameter error
empty_profile = {}
query = "How often must reviews happen?"
docs = [{
    "id": "NIST_SP800-53:rev5:AC-2",
    "text": "The organization reviews accounts {{ insert: param, ac-02_odp.05 }}."
}]

res = generator.generate(query, docs, empty_profile)
pprint(res)

if res.get('status') == 'PARAMS_REQUIRED' and 'ac-02_odp.05' in res.get('odp_required_list', []):
    print(" SUCCESS: System halted and correctly flagged missing parameter.")
else:
    print(f" FAILURE: System did not halt. Status: {res.get('status')}")

print("\n Evaluation Complete.")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



TEST A: NIST Rev 5 (Substitution Check)
{'answer_text': 'The organization reviews accounts for compliance 90 days.',
 'evidence_spans': [{'source_id': 'NIST_SP800-53:rev5:AC-2',
                     'span_text': 'The organization reviews accounts for '
                                  'compliance {{ insert: param, ac-02_odp.05 '
                                  '}}.'}],
 'odp_required_list': [],
 'status': 'OK'}
 SUCCESS: Parameter substituted correctly.


TEST B: NIST Rev 4 (Legacy Check)
{'answer_text': 'Develops, documents, and disseminates to Security Officer the '
                'policy.',
 'evidence_spans': [{'source_id': 'NIST_SP800-53:rev4:AC-1',
                     'span_text': 'Develops, documents, and disseminates to {{ '
                                  'insert: param, ac-1_prm_1 }} the policy.'}],
 'odp_required_list': [],
 'status': 'OK'}
 SUCCESS: Parameter substituted correctly.


TEST C: Missing Data (Traffic Cop Check)
{'answer_text': 'The organization reviews a